In [1]:
import pandas as pd
import numpy as np
from decimal import Decimal
import glob
import backtrader as bt
import matplotlib
import matplotlib.pyplot as plt
import pandas_ta as ta
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Ensure plots are rendered inline in the notebook
pio.renderers.default = "notebook"
matplotlib.use('Agg')
%matplotlib inline

In [2]:
# Define the path to your CSV files
file_path_pattern = '/root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_*.csv'

# Define the column names
column_names = ['DATE', 'TIME', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']

# Load and combine all CSV files
all_files = glob.glob(file_path_pattern)
data_list = []

for file in all_files:
    print(f"Loading file: {file}")  # Debugging: Print file being loaded
    df = pd.read_csv(file, delimiter=',', names=column_names, header=None, dtype=str)
    # print(df.head())  # Debugging: Print the first few rows of the loaded DataFrame
    data_list.append(df)

combined_data = pd.concat(data_list)

# Check the combined data before parsing dates
print("Combined data before parsing dates:")
print(combined_data.head())

# Combine <DATE> and <TIME> into a single datetime column
combined_data['datetime'] = pd.to_datetime(combined_data['DATE'] + ' ' + combined_data['TIME'], format='%Y.%m.%d %H:%M', errors='coerce')

# Check the combined data after parsing dates
print("Combined data after parsing dates:")
print(combined_data.head())

# Check for rows with NaT in datetime column
print("Rows with NaT in datetime column:")
print(combined_data[combined_data['datetime'].isna()].head())

# Drop rows with NaT in datetime column
combined_data.dropna(subset=['datetime'], inplace=True)

# Set datetime as the index
combined_data.set_index('datetime', inplace=True)
combined_data.sort_index(inplace=True)

# Check the combined data before converting numeric columns
print("Combined data before converting numeric columns:")
print(combined_data.head())

# Convert numeric columns to appropriate data types
numeric_columns = ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']
combined_data[numeric_columns] = combined_data[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Check the combined data after converting numeric columns
print("Combined data after converting numeric columns:")
print(combined_data.head())

# Drop rows with NaN values in numeric columns
combined_data.replace([np.inf, -np.inf], np.nan, inplace=True)
combined_data.fillna(0, inplace=True)

# Display the final combined data
print("Final combined data:")
combined_data.head()

Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2007.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2018.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2005.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2012.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2000.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2010.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2013.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2015.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/EURUSD/DAT_MT_EURUSD_M1_2017.csv
L

,DATE,TIME,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD
datetime,,,,,,,,,
2000-05-30 17:27:00,2000.05.30,17:27,0.9302,0.9302,0.9302,0.9302,0,0.0,0.0
2000-05-30 17:35:00,2000.05.30,17:35,0.9304,0.9305,0.9304,0.9305,0,0.0,0.0
2000-05-30 17:38:00,2000.05.30,17:38,0.9304,0.9304,0.9303,0.9303,0,0.0,0.0
2000-05-30 17:43:00,2000.05.30,17:43,0.9301,0.9301,0.9300,0.9300,0,0.0,0.0
2000-05-30 17:44:00,2000.05.30,17:44,0.9298,0.9298,0.9297,0.9297,0,0.0,0.0


In [3]:
combined_data_1m = combined_data

combined_data_1m = combined_data_1m.drop(columns=['DATE', 'TIME'])

combined_data_1m.head()

,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD
datetime,,,,,,,
2000-05-30 17:27:00,0.9302,0.9302,0.9302,0.9302,0,0.0,0.0
2000-05-30 17:35:00,0.9304,0.9305,0.9304,0.9305,0,0.0,0.0
2000-05-30 17:38:00,0.9304,0.9304,0.9303,0.9303,0,0.0,0.0
2000-05-30 17:43:00,0.9301,0.9301,0.9300,0.9300,0,0.0,0.0
2000-05-30 17:44:00,0.9298,0.9298,0.9297,0.9297,0,0.0,0.0


In [4]:
# Calculate indicators
aroon_length = 50
ema_length = 20
stoch_rsi_length = 14
stoch_rsi_smoothK = 3
stoch_rsi_smoothD = 3
supertrend_length = 50
supertrend_mult = 3.0
kc_length = 20
kc_scalar = 1.5
atr_length = 14

combined_data_1m['AROON-OSC'] = ta.aroon(
    combined_data_1m['HIGH'],
    combined_data_1m['LOW'],
    length=aroon_length
)[f'AROONOSC_{aroon_length}'].round(2)
combined_data_1m['EMA_50'] = ta.ema(combined_data_1m['CLOSE'], length=ema_length).round(5)
combined_data_1m['STOCH-RSId'] = ta.stochrsi(
    combined_data_1m['CLOSE'],
    length=stoch_rsi_length,
    smoothK=stoch_rsi_smoothK,
    smoothD=stoch_rsi_smoothD
)[f'STOCHRSId_{stoch_rsi_length}_{stoch_rsi_length}_{stoch_rsi_smoothK}_{stoch_rsi_smoothD}'].round(2)
combined_data_1m['STOCH-RSIk'] = ta.stochrsi(
    combined_data_1m['CLOSE'],
    length=stoch_rsi_length,
    smoothK=stoch_rsi_smoothK,
    smoothD=stoch_rsi_smoothD
)[f'STOCHRSIk_{stoch_rsi_length}_{stoch_rsi_length}_{stoch_rsi_smoothK}_{stoch_rsi_smoothD}'].round(2)
combined_data_1m['SUPERTREND'] = ta.supertrend(
    combined_data_1m['HIGH'],
    combined_data_1m['LOW'],
    combined_data_1m['CLOSE'],
    length=supertrend_length,
    multiplier=supertrend_mult
)[f'SUPERT_{supertrend_length}_{supertrend_mult}'].round(5)
combined_data_1m['Keltner-Upper'] = ta.kc(
    high=combined_data_1m['HIGH'],
    low=combined_data_1m['LOW'],
    close=combined_data_1m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCUe_{kc_length}_{kc_scalar}'].round(5)
combined_data_1m['Keltner-Basis'] = ta.kc(
    high=combined_data_1m['HIGH'],
    low=combined_data_1m['LOW'],
    close=combined_data_1m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCBe_{kc_length}_{kc_scalar}'].round(5)
combined_data_1m['Keltner-Lower'] = ta.kc(
    high=combined_data_1m['HIGH'],
    low=combined_data_1m['LOW'],
    close=combined_data_1m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCLe_{kc_length}_{kc_scalar}'].round(5)
combined_data_1m['ATR'] = ta.atr(
    high=combined_data_1m['HIGH'],
    low=combined_data_1m['LOW'],
    close=combined_data_1m['CLOSE'],
    length=atr_length
).round(5)



# Drop rows with NaN values (due to indicator calculation)
combined_data_1m.dropna(inplace=True)

# Display the data with indicators
print("Data with indicators:")
combined_data_1m.head()

Data with indicators:


,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD,AROON-OSC,EMA_50,STOCH-RSId,STOCH-RSIk,SUPERTREND,Keltner-Upper,Keltner-Basis,Keltner-Lower,ATR
datetime,,,,,,,,,,,,,,,,
2000-05-30 19:56:00,0.9297,0.9297,0.9297,0.9297,0,0.0,0.0,-78.0,0.92961,80.49,78.19,0.92930,0.92979,0.92961,0.92943,0.00012
2000-05-30 19:57:00,0.9296,0.9296,0.9295,0.9295,0,0.0,0.0,-78.0,0.92960,78.62,79.47,0.92930,0.92979,0.92960,0.92941,0.00013
2000-05-30 19:58:00,0.9297,0.9299,0.9297,0.9298,0,0.0,0.0,-76.0,0.92962,81.34,86.36,0.92938,0.92985,0.92962,0.92939,0.00015
2000-05-30 19:59:00,0.9299,0.9305,0.9299,0.9305,0,0.0,0.0,26.0,0.92970,84.07,86.36,0.92974,0.93001,0.92970,0.92939,0.00019
2000-05-30 20:00:00,0.9306,0.9307,0.9306,0.9307,0,0.0,0.0,28.0,0.92980,90.91,100.00,0.93019,0.93010,0.92980,0.92949,0.00019


In [64]:
plotly_data = combined_data_1m.loc['2021-01-01':'2021-01-31']

# Create a Plotly figure with subplots
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.02, 
                    row_heights=[0.10, 0.75, 0.15, 0.15],
                    subplot_titles=('ATR 50',
                                    'EMA 50 | Supertrend 50, 3.0 | Keltner Channels 20, 1.5',
                                    'Aroon Oscillator 50',
                                    'Stochastic RSI 14, 3, 3'
                                )
                            )

# Add candlestick chart
fig.add_trace(go.Candlestick(x=plotly_data.index,
                             open=plotly_data['OPEN'],
                             high=plotly_data['HIGH'],
                             low=plotly_data['LOW'],
                             close=plotly_data['CLOSE'],
                             ),
            row=2, col=1)

# Add ATR to top of chart
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['ATR'],
    mode='lines',
    line=go.scatter.Line(color='yellow'),
    name='ATR'
), row=1, col=1)
# Add indicators to candlestick chart
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['EMA_50'],
    mode='lines',
    line=go.scatter.Line(color='red'),
    name='EMA 50'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['SUPERTREND'],
    mode='lines',
    line=go.scatter.Line(color='green'),
    name='SUPERTREND'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Upper'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Keltner-Upper'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Basis'],
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Keltner-Basis'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Lower'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Keltner-Lower'
), row=2, col=1)

# Add Aroon Oscillator
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['AROON-OSC'],
    mode='lines',
    line=go.scatter.Line(color='purple'),
    name='Aroon Oscillator'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[0] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='grey', dash='dash'),
    name='Aroon Osc Zero Line'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[60] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Aroon Osc Mid Line'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[-60] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Aroon Osc Upper Line'
), row=3, col=1)

# Add Stochastic RSI
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['STOCH-RSId'],
    mode='lines',
    line=go.scatter.Line(color='orange'),
    name='Stochastic RSI %D'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['STOCH-RSIk'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Stochastic RSI %K'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[20] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='red', dash='dash'),
    name='Oversold Threshold'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[80] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='green', dash='dash'),
    name='Overbought Threshold'
), row=4, col=1)


fig.update_layout(
    title='EURUSD 1m Candlestick Chart with Indicators January 2021',
    xaxis2=dict(
        rangeslider=dict(visible=False),
        rangebreaks=[
            dict(bounds=["sat", "mon"]),  # hide weekends
            dict(values=["2021-01-08 17:00", "2021-01-10 17:00"])
        ]
    ),
    yaxis=dict(title='Price', autorange=True),
    dragmode='zoom',
    uirevision='dynamic',
)

fig.write_html('./plotly/test_plot_EURUSD_1m.html')

In [5]:
combined_data_5m = combined_data.resample('5min').agg({
    'OPEN': 'first',
    'HIGH': 'max',
    'LOW': 'min',
    'CLOSE': 'last',
    'TICKVOL': 'sum',  # Aggregate tick volumes
    'VOL': 'sum',      # Aggregate actual volumes
    'SPREAD': 'mean',  # Average spread
}).dropna()

# Ensure datetime is set as the index
combined_data_5m.reset_index(inplace=True)  # Ensure 'datetime' is a column
combined_data_5m['datetime'] = pd.to_datetime(combined_data_5m['datetime'])  # Convert to datetime
combined_data_5m.set_index('datetime', inplace=True)  # Set as index

# Display the resampled data
combined_data_5m.head()

,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD
datetime,,,,,,,
2000-05-30 17:25:00,0.9302,0.9302,0.9302,0.9302,0,0.0,0.0
2000-05-30 17:35:00,0.9304,0.9305,0.9303,0.9303,0,0.0,0.0
2000-05-30 17:40:00,0.9301,0.9301,0.9297,0.9297,0,0.0,0.0
2000-05-30 18:25:00,0.9298,0.9299,0.9298,0.9299,0,0.0,0.0
2000-05-30 18:35:00,0.9300,0.9300,0.9300,0.9300,0,0.0,0.0


In [6]:
# Calculate indicators
aroon_length = 50
ema_length = 20
stoch_rsi_length = 14
stoch_rsi_smoothK = 3
stoch_rsi_smoothD = 3
supertrend_length = 50
supertrend_mult = 3.0
kc_length = 20
kc_scalar = 1.5
atr_length = 14

combined_data_5m['AROON-OSC'] = ta.aroon(
    combined_data_5m['HIGH'],
    combined_data_5m['LOW'],
    length=aroon_length
)[f'AROONOSC_{aroon_length}'].round(2)
combined_data_5m['EMA_50'] = ta.ema(combined_data_5m['CLOSE'], length=ema_length).round(5)
combined_data_5m['STOCH-RSId'] = ta.stochrsi(
    combined_data_5m['CLOSE'],
    length=stoch_rsi_length,
    smoothK=stoch_rsi_smoothK,
    smoothD=stoch_rsi_smoothD
)[f'STOCHRSId_{stoch_rsi_length}_{stoch_rsi_length}_{stoch_rsi_smoothK}_{stoch_rsi_smoothD}'].round(2)
combined_data_5m['STOCH-RSIk'] = ta.stochrsi(
    combined_data_5m['CLOSE'],
    length=stoch_rsi_length,
    smoothK=stoch_rsi_smoothK,
    smoothD=stoch_rsi_smoothD
)[f'STOCHRSIk_{stoch_rsi_length}_{stoch_rsi_length}_{stoch_rsi_smoothK}_{stoch_rsi_smoothD}'].round(2)
combined_data_5m['SUPERTREND'] = ta.supertrend(
    combined_data_5m['HIGH'],
    combined_data_5m['LOW'],
    combined_data_5m['CLOSE'],
    length=supertrend_length,
    multiplier=supertrend_mult
)[f'SUPERT_{supertrend_length}_{supertrend_mult}'].round(5)
combined_data_5m['Keltner-Upper'] = ta.kc(
    high=combined_data_5m['HIGH'],
    low=combined_data_5m['LOW'],
    close=combined_data_5m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCUe_{kc_length}_{kc_scalar}'].round(5)
combined_data_5m['Keltner-Basis'] = ta.kc(
    high=combined_data_5m['HIGH'],
    low=combined_data_5m['LOW'],
    close=combined_data_5m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCBe_{kc_length}_{kc_scalar}'].round(5)
combined_data_5m['Keltner-Lower'] = ta.kc(
    high=combined_data_5m['HIGH'],
    low=combined_data_5m['LOW'],
    close=combined_data_5m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCLe_{kc_length}_{kc_scalar}'].round(5)
combined_data_5m['ATR'] = ta.atr(
    high=combined_data_5m['HIGH'],
    low=combined_data_5m['LOW'],
    close=combined_data_5m['CLOSE'],
    length=atr_length
).round(5)


# Drop rows with NaN values (due to indicator calculation)
combined_data_5m.dropna(inplace=True)

# Display the data with indicators
print("5 min Data with indicators:")
combined_data_5m.head()

5 min Data with indicators:


,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD,AROON-OSC,EMA_50,STOCH-RSId,STOCH-RSIk,SUPERTREND,Keltner-Upper,Keltner-Basis,Keltner-Lower,ATR
datetime,,,,,,,,,,,,,,,,
2000-05-31 01:00:00,0.9318,0.9318,0.9318,0.9318,0,0.0,0.0,30.0,0.93155,35.15,40.78,0.93096,0.93189,0.93155,0.93121,0.00025
2000-05-31 01:05:00,0.9317,0.9319,0.9317,0.9319,0,0.0,0.0,30.0,0.93158,41.57,50.38,0.93096,0.93192,0.93158,0.93124,0.00024
2000-05-31 01:10:00,0.9321,0.9322,0.9321,0.9321,0,0.0,0.0,30.0,0.93163,50.26,59.60,0.93131,0.93198,0.93163,0.93128,0.00025
2000-05-31 01:15:00,0.9322,0.9322,0.9322,0.9322,0,0.0,0.0,30.0,0.93169,59.53,68.62,0.93137,0.93202,0.93169,0.93136,0.00024
2000-05-31 01:30:00,0.9317,0.9317,0.9317,0.9317,0,0.0,0.0,30.0,0.93169,62.62,59.63,0.93137,0.93206,0.93169,0.93132,0.00025


In [7]:
data_5m_2018 = combined_data_5m.loc['2018-01-01':'2018-12-31']

print(data_5m_2018.head())

matching_rows = data_5m_2018[data_5m_2018['CLOSE'] == data_5m_2018['SUPERTREND']]
print(f'Number of matching rows: {len(matching_rows)}')
matching_rows.head()

                        OPEN     HIGH      LOW    CLOSE  TICKVOL  VOL  SPREAD  \
datetime                                                                        
2018-01-01 17:00:00  1.20037  1.20100  1.20017  1.20048        0  0.0     0.0   
2018-01-01 17:05:00  1.20050  1.20097  1.20027  1.20094        0  0.0     0.0   
2018-01-01 17:10:00  1.20094  1.20095  1.20076  1.20093        0  0.0     0.0   
2018-01-01 17:15:00  1.20092  1.20105  1.20036  1.20050        0  0.0     0.0   
2018-01-01 17:20:00  1.20053  1.20055  1.20044  1.20048        0  0.0     0.0   

                     AROON-OSC   EMA_50  STOCH-RSId  STOCH-RSIk  SUPERTREND  \
datetime                                                                      
2018-01-01 17:00:00      -92.0  1.20008       70.37       94.04     1.19961   
2018-01-01 17:05:00      -92.0  1.20016       87.19      100.00     1.19962   
2018-01-01 17:10:00      -92.0  1.20023       97.95       99.80     1.19986   
2018-01-01 17:15:00      -86.0  1.200

,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD,AROON-OSC,EMA_50,STOCH-RSId,STOCH-RSIk,SUPERTREND,Keltner-Upper,Keltner-Basis,Keltner-Lower,ATR
datetime,,,,,,,,,,,,,,,,
2018-01-02 00:40:00,1.20193,1.20206,1.20187,1.20206,0,0.0,0.0,-32.0,1.20173,99.09,99.98,1.20206,1.20196,1.20173,1.20150,0.00017
2018-01-18 04:05:00,1.21978,1.21997,1.21970,1.21970,0,0.0,0.0,26.0,1.22049,0.61,0.00,1.21970,1.22143,1.22049,1.21955,0.00063
2018-01-26 14:40:00,1.24245,1.24251,1.24176,1.24232,0,0.0,0.0,40.0,1.24339,17.41,12.26,1.24232,1.24460,1.24339,1.24218,0.00079
2018-01-31 11:00:00,1.24563,1.24631,1.24522,1.24529,0,0.0,0.0,34.0,1.24607,29.28,10.16,1.24529,1.24727,1.24607,1.24487,0.00076
2018-02-12 02:30:00,1.22861,1.22880,1.22780,1.22807,0,0.0,0.0,-62.0,1.22860,48.65,19.46,1.22807,1.22943,1.22860,1.22776,0.00053


In [63]:
plotly_data = combined_data_5m.loc['2021-01-01':'2021-01-31']

# Create a Plotly figure with subplots
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.04, 
                    row_heights=[0.10, 0.75, 0.15, 0.15],
                    subplot_titles=(
                        'ATR 14',
                        'EMA 50 | Supertrend 50, 3.0 | Keltner Channels 20, 1.5',
                        'Aroon Oscillator 50',
                        'Stochastic RSI 14, 3, 3'
                    )
                )

# Add candlestick chart
fig.add_trace(go.Candlestick(x=plotly_data.index,
                             open=plotly_data['OPEN'],
                             high=plotly_data['HIGH'],
                             low=plotly_data['LOW'],
                             close=plotly_data['CLOSE'],
                             ),
            row=2, col=1)

# Add ATR to top of chart
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['ATR'],
    mode='lines',
    line=go.scatter.Line(color='yellow'),
    name='ATR'
), row=1, col=1)
# Add indicators to candlestick chart
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['EMA_50'],
    mode='lines',
    line=go.scatter.Line(color='red'),
    name='EMA 50'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['SUPERTREND'],
    mode='lines',
    line=go.scatter.Line(color='green'),
    name='SUPERTREND'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Upper'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Keltner-Upper'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Basis'],
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Keltner-Basis'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Lower'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Keltner-Lower'
), row=2, col=1)

# Add Aroon Oscillator
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['AROON-OSC'],
    mode='lines',
    line=go.scatter.Line(color='purple'),
    name='Aroon Oscillator'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[0] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='grey', dash='dash'),
    name='Aroon Osc Zero Line'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[60] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Aroon Osc Mid Line'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[-60] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Aroon Osc Upper Line'
), row=3, col=1)

# Add Stochastic RSI
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['STOCH-RSId'],
    mode='lines',
    line=go.scatter.Line(color='orange'),
    name='Stochastic RSI %D'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['STOCH-RSIk'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Stochastic RSI %K'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[20] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='red', dash='dash'),
    name='Oversold Threshold'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[80] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='green', dash='dash'),
    name='Overbought Threshold'
), row=4, col=1)


fig.update_layout(
    title='EURUSD 5m Candlestick Chart with Indicators January 2021',
    xaxis2=dict(
        rangeslider=dict(visible=False),
        rangebreaks=[
            dict(bounds=["sat", "mon"]),  # hide weekends
            dict(values=["2021-01-08 17:00", "2021-01-10 17:00"])
        ]
    ),
    yaxis=dict(title='Price', autorange=True),
    dragmode='zoom',
    uirevision='dynamic',
)

fig.write_html('./plotly/test_plot_EURUSD_5m.html')

In [23]:
symbol = "EURUSD"
year = 2019
end_year = 2019

In [26]:
class SuperTrendStrategy(bt.Strategy):
    params = dict(
        risk_per_trade=0.0025,
        rr_ratio=2.8,
    )

    def __init__(self):
        self.buy_order = None
        self.sell_order = None
        self.trade_log = []
        self.open_trades = {}
        # self.data_1m = self.datas[0]
        # self.data_5m = self.datas[1]
        # Indicators
        self.supertrend = self.data.supertrend
        self.ema = self.data.ema_50
        self.aroon = self.data.aroon_osc
        self.stoch_rsi_d = self.data.stoch_rsi_d
        self.stoch_rsi_k = self.data.stoch_rsi_k
        self.kc_upper = self.data.keltner_upper
        self.kc_basis = self.data.keltner_basis
        self.kc_lower = self.data.keltner_lower
        self.atr = self.data.atr
        

    def log(self, *args, dt=None):
        '''Logging function for this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), ', '.join(map(str, args))))
    
    def calculate_take_profit(self, current_price, stop_loss, is_buy):
            if is_buy:
                return current_price + ((current_price - stop_loss) * self.params.rr_ratio)
            else:
                return current_price - ((stop_loss - current_price) * self.params.rr_ratio)

    def calculate_volume(self, current_price, stop_loss):
        # Calculate position size based on risk
        account_balance = self.broker.getvalue()
        risk_amount = account_balance * self.params.risk_per_trade
        stop_loss_distance = abs(current_price - stop_loss)

        # Ensure stop loss distance is not zero or too small
        if stop_loss_distance == 0:
            self.log("Skipping trade due to stop loss distance being 0")
            return None

        # Calculate volume in lots (1 lot = 100,000 units)
        volume = risk_amount / stop_loss_distance / 100000
        # if volume < 0:
        #     self.log(f"Skipping trade due to invalid volume - Volume: {volume}")
        #     return None
        # self.log(f"Calculated volume: {volume:.2f} lots")
        return round(volume, 2)           

    def next(self):
        if self.position:
            return
        
        # Fetch values
        current_price = self.data.close[0]
        current_high_range = self.data.high.get(size=5)
        current_low_range = self.data.low.get(size=5)
        recent_highs = self.data.high.get(size=10)
        recent_lows = self.data.low.get(size=10)

        if len(current_high_range) < 5 or len(current_low_range) < 5:
            self.log("Not enough data points to calculate current high and low.")
            return
        current_high = max(current_high_range)
        current_low = min(current_low_range)


        # Check if there are enough data points
        if len(recent_highs) < 10 or len(recent_lows) < 10:
            self.log("Not enough data points to calculate recent high and low.")
            return
        recent_high = max(recent_highs)
        recent_low = min(recent_lows)
        
        recent_high_index = self.data.high.get(size=10).index(recent_high)
        recent_low_index = self.data.low.get(size=10).index(recent_low)

        ago_recent_high = -(10 - recent_high_index)
        ago_recent_low = -(10 - recent_low_index)

        kc_upper_at_recent_high = self.kc_upper[ago_recent_high]
        kc_lower_at_recent_low = self.kc_lower[ago_recent_low]
        kc_basis = self.kc_basis[0]
        atr = self.atr[0]
        supertrend = self.supertrend[0]
        ema = self.ema[0]
        aroon = self.aroon[0]
        stoch_k = self.stoch_rsi_k[0]
        stoch_k_recent_max = max(self.stoch_rsi_k.get(size=5))
        stoch_k_recent_min = min(self.stoch_rsi_k.get(size=5))
        stoch_d = self.stoch_rsi_d[0]

        # Set Aroon Oscillator threshold
        aroon_threshold = 40

        # Log indicator values
        # print(f"Current Price: {current_price:.5f}, Supertrend: {supertrend:.5f}, EMA: {ema:.5f}, Aroon: {aroon:.2f}, Stoch K: {stoch_k:.2f}, Stoch D: {stoch_d:.2f}")

        # Calculate stop loss and take profit
        stop_loss = round(supertrend, 5)
        take_profit_buy = self.calculate_take_profit(current_price, stop_loss, is_buy=True)
        take_profit_sell = self.calculate_take_profit(current_price, stop_loss, is_buy=False)
        volume = self.calculate_volume(current_price, stop_loss)

        if volume is None or volume <= 0:
            self.log("Volume calculation resulted in None or non-positive value.")
            return
        
        
        # Long entry
        if (
            not self.buy_order 
            and not self.sell_order
            and current_price > supertrend 
            and current_price > ema
            and recent_high > kc_upper_at_recent_high
            and current_low <= kc_basis 
            and aroon > aroon_threshold 
            and stoch_k_recent_min < 30 
            and stoch_d < stoch_k
        ):
            main_order = self.buy(size=abs(volume), exectype=bt.Order.Market, transmit=False)
            stop_order = self.sell(
                size=abs(volume), exectype=bt.Order.Stop,
                price=stop_loss, parent=main_order, transmit=False
            )
            profit_order = self.sell(
                size=abs(volume), exectype=bt.Order.Limit,
                price=take_profit_buy, parent=main_order, transmit=True
            )
            self.buy_order = main_order

        # Short entry
        elif (
            not self.sell_order
            and not self.buy_order
            and current_price < supertrend 
            and current_price < ema 
            and recent_low < kc_lower_at_recent_low
            and current_high >= kc_basis
            and aroon < -aroon_threshold
            and stoch_k_recent_max > 70 
            and stoch_d > stoch_k
        ):
            main_order = self.sell(size=abs(volume), exectype=bt.Order.Market, transmit=False)
            stop_order = self.buy(
                size=abs(volume), exectype=bt.Order.Stop,
                price=stop_loss, parent=main_order, transmit=False
            )
            profit_order = self.buy(
                size=abs(volume), exectype=bt.Order.Limit,
                price=take_profit_sell, parent=main_order, transmit=True
            )
            self.sell_order = main_order

    def log_trade(self, trade_info):
        """Log a trade into the trade log."""
        self.trade_log.append(trade_info)

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return

        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f"BUY EXECUTED {order.ref} Price: \
{order.executed.price:.5f} \
Size: {order.executed.size:.2f}")
                self.buy_order = None  # Clear buy order
                self.open_trades[order.ref] = {
                    "TRADE NUMBER": len(self.trade_log) + 1,
                    "SYMBOL": "EURUSD",
                    "OPEN TIME": self.data.datetime.datetime(0),
                    "VOLUME": self.calculate_volume(self.data.close[0], self.supertrend[0]),
                    "SIDE": "BUY",
                    "OPEN PRICE": round(order.executed.price, 5),
                    "STOP LOSS": round(self.supertrend[0], 5),
                    "TAKE PROFIT": round(self.calculate_take_profit(self.data.close[0], self.supertrend[0], is_buy=True), 5),
                }
                self.log_trade(self.open_trades[order.ref])
            elif order.issell():
                self.log(f"SELL EXECUTED {order.ref} Price: {order.executed.price:.5f} Size: {order.executed.size:.2f}")
                self.sell_order = None  # Clear sell order
                self.open_trades[order.ref] = {
                    "TRADE NUMBER": len(self.trade_log) + 1,
                    "SYMBOL": "EURUSD",
                    "OPEN TIME": self.data.datetime.datetime(0),
                    "VOLUME": self.calculate_volume(self.data.close[0], self.supertrend[0]),
                    "SIDE": "SELL",
                    "OPEN PRICE": round(order.executed.price, 5),
                    "STOP LOSS": round(self.supertrend[0], 5),
                    "TAKE PROFIT": round(self.calculate_take_profit(self.data.close[0], self.supertrend[0], is_buy=False), 5),
                    "CLOSE TIME": None,
                    "CLOSE PRICE": None,
                    "PROFIT": None,
                    "RUNNING BALANCE": None,
                }
                self.log_trade(self.open_trades[order.ref])

            # Check if the order is closing an existing trade
            if order.exectype in [bt.Order.Stop, bt.Order.Limit] and order.parent:
                parent_ref = order.parent.ref
                if parent_ref in self.open_trades:
                    trade_info = self.open_trades.pop(parent_ref, {})
                    trade_info.update({
                        "CLOSE TIME": self.data.datetime.datetime(0),
                        "CLOSE PRICE": round(order.executed.price, 5),
                        "PROFIT": round(order.executed.pnl, 2),
                        "RUNNING BALANCE": round(self.broker.getvalue(), 2),
                    })
                    self.log_trade(trade_info)
                    self.log(f"TRADE CLOSED {parent_ref} by {order.ref}, Gross PnL={order.executed.pnl:.2f}")

            self.bar_executed = len(self)

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log(f'Order Canceled/Margin/Rejected: {order.ref}, Status: {order.getstatusname()}')
            if order.status == order.Margin:
                self.log(f"Margin call: Not enough margin for order {order.ref}")
            elif order.status == order.Rejected:
                self.log(f"Order rejected: {order.ref}")

        self.order = None

    # def notify_trade(self, trade):
    #     if not trade.isclosed:
    #         return

    #     self.log(f"TRADE CLOSED trade.myref={final_ref}, Gross PnL={trade.pnl:.2f}")

    #     trade_info = self.open_trades.pop(final_ref, {})
        
    #     trade_info.update({
    #         "CLOSE TIME": self.data.datetime.datetime(0),
    #         "CLOSE PRICE": round(trade.price, 5),
    #         "PROFIT": round(trade.pnl, 2),
    #         "RUNNING BALANCE": round(self.broker.getvalue(), 2),
    #     })
    #     self.log_trade(trade_info)

    def stop(self):
        # Save trades to a DataFrame at the end of the backtest
        self.trade_df = pd.DataFrame(self.trade_log)

        # Filter out canceled orders
        self.trade_df = self.trade_df[self.trade_df['CLOSE PRICE'].notna()]

        # Remove duplicates
        self.trade_df = self.trade_df.drop_duplicates()

        self.trade_df.to_csv(
            f'./back_test_data/{symbol}_{year}_5m_ST{supertrend_length}_EMA{ema_length}_\
AROON{aroon_length}.csv',
            index=False
        )

In [27]:
# Convert the pandas DataFrame to a Backtrader data feed
class PandasData(bt.feeds.PandasData):
    lines = (
        'ema_50',
        'stoch_rsi_k',
        'stoch_rsi_d',
        'supertrend',
        'aroon_osc',
        'keltner_upper',
        'keltner_basis',
        'keltner_lower',
        'atr',
    )
    params = (
        ('datetime', None),
        ('open', 'OPEN'),
        ('high', 'HIGH'),
        ('low', 'LOW'),
        ('close', 'CLOSE'),
        ('volume', 'VOL'),
        ('ema_50', 'EMA_50'),
        ('stoch_rsi_k', 'STOCH-RSIk'),
        ('stoch_rsi_d', 'STOCH-RSId'),
        ('supertrend', 'SUPERTREND'),
        ('aroon_osc', 'AROON-OSC'),
        ('keltner_upper', 'Keltner-Upper'),
        ('keltner_basis', 'Keltner-Basis'),
        ('keltner_lower', 'Keltner-Lower'),
        ('atr', 'ATR'),
    )

test_data_1m = combined_data_1m.loc[f'{year}-01-01':f'{end_year}-12-31']
test_data_5m = combined_data_5m.loc[f'{year}-01-01':f'{end_year}-12-31']

data_feed = PandasData(dataname=test_data_5m)

comm_info = bt.CommInfoBase(
    commission=0.02,
    leverage=30,
    margin=1 / 30,
    mult=100000
)

# Set up the Backtrader environment
cerebro = bt.Cerebro()
cerebro.addstrategy(SuperTrendStrategy)
cerebro.adddata(data_feed)
# cerebro.resampledata(data_feed, timeframe=bt.TimeFrame.Minutes, compression=5, name='5m')
cerebro.broker.set_cash(2500)
cerebro.broker.addcommissioninfo(comm_info)

# Run the backtest
cerebro.run(style='candlestick')
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())
# print('Number of trades: %d' % len())
# print('Max Drawdown: %.2f' % cerebro.broker.getdrawdown().max.drawdown)
# Plot the results
# cerebro.plot(show=True)
# Dates to plot
# from_date = '2018-12-21'
# to_date = '2018-12-31'
# fig = cerebro.plot(
#     style='candelstick',
#     volume=False,
#     fromdate=pd.Timestamp(from_date),
#     todate=pd.Timestamp(to_date)
# )[0][0]
# fig.savefig('backtrader_plot_5m_2018_aroon24_ema50_st50.pdf', format='pdf', dpi=300)


2019-01-01, Not enough data points to calculate current high and low.
2019-01-01, Not enough data points to calculate current high and low.
2019-01-01, Not enough data points to calculate current high and low.
2019-01-01, Not enough data points to calculate current high and low.
2019-01-01, Not enough data points to calculate recent high and low.
2019-01-01, Not enough data points to calculate recent high and low.
2019-01-01, Not enough data points to calculate recent high and low.
2019-01-01, Not enough data points to calculate recent high and low.
2019-01-01, Not enough data points to calculate recent high and low.
2019-01-01, BUY EXECUTED 14667 Price: 1.14619 Size: 0.08
2019-01-01, SELL EXECUTED 14668 Price: 1.14538 Size: -0.08
2019-01-01, TRADE CLOSED 14667 by 14668, Gross PnL=-6.48
2019-01-01, Order Canceled/Margin/Rejected: 14669, Status: Canceled
2019-01-01, SELL EXECUTED 14670 Price: 1.14521 Size: -0.15
2019-01-02, BUY EXECUTED 14671 Price: 1.14562 Size: 0.15
2019-01-02, TRADE 